# Forecasting Daily High Temperatures in Montreal for Energy Demand Prediction

## Project Description
This project focuses on building and evaluating machine learning models to forecast daily high temperatures in Montreal. Accurate temperature predictions are crucial for energy demand forecasting, which aids in resource allocation and grid management. The project employs a time-series approach, including robust data preprocessing, feature engineering, and model training using Linear Regression, Neural Networks, and Support Vector Machines (SVMs).

## Dataset
The dataset used for this project consists of weather station records from 42 stations in Montreal, spanning the years 2016-2025. It includes the following columns:
- `STATION`: Weather station identifier
- `NAME`: Name of the weather station
- `DATE`: Date of observation
- `DAPR`: Days with precipitation
- `MDPR`: Maximum daily precipitation
- `PRCP`: Precipitation
- `SNOW`: Snowfall
- `SNWD`: Snow depth
- `TAVG`: Average temperature
- `TMAX`: Maximum temperature
- `TMIN`: Minimum temperature
- `WDFG`: Wind direction (fastest gust)
- `WESD`: Water equivalent of snow on ground
- `WESF`: Water equivalent of snowfall
- `WSFG`: Wind speed (fastest gust)

## Data Preprocessing and Feature Engineering

1.  **Daily Aggregation**: Numeric columns were grouped by `DATE`, and their means were calculated to create a daily aggregated dataset. Columns with over 90% missing values (`DAPR`, `MDPR`) were dropped.
2.  **Date Feature Engineering**: Extracted `year`, `month`, `day_of_year`, and `week` from the `DATE` column. Cyclical transformations (sine and cosine) were applied to `month` and `day_of_year` to capture seasonality.
3.  **Derived Features**: Calculated `temp_range` (`TMAX` - `TMIN`), `snow_flag` (1 if `SNOW` > 0), and `prcp_flag` (1 if `PRCP` > 0).
4.  **Missing Value Imputation**: A three-pass imputation strategy was used:
    -   **Pass 1**: Linear interpolation (time-aware) for short gaps.
    -   **Pass 2**: Seasonal median imputation for remaining gaps, using the median value of the same month across surrounding years.
    -   **Pass 3**: Global median fallback for any persistent missing values.
5.  **Target Definition**: The `TMAX` column was shifted forward by 1 day to create `TMAX_tomorrow`, serving as the prediction target. This simulates real-world forecasting where today's data predicts tomorrow's high temperature.
6.  **Outlier Clipping**: Features were clipped between the 1st and 99th percentiles to handle extreme outliers, ensuring robustness of the models.

## Walk-Forward Cross-Validation
An expanding-window, walk-forward cross-validation strategy was implemented to evaluate model performance on future, unseen data. The dataset was split as follows:
-   **Fold 1**: Train [2016-2020] → Validate [2021]
-   **Fold 2**: Train [2016-2021] → Validate [2022]
-   **Fold 3**: Train [2016-2022] → Validate [2023]
-   **Fold 4**: Train [2016-2023] → Validate [2024]
-   **Final Evaluation**: Train [2016-2024] → Test [2025]

Features were scaled using `StandardScaler`, with each fold fitting its own scaler to prevent data leakage.

## Model Architectures

### 1. Linear Model (PyTorch)
-   **Architecture**: A simple linear regression model (`nn.Linear`) from PyTorch.
-   **Loss Functions**: Mean Squared Error (MSE), Root Mean Squared Error (RMSE), Huber Loss, and Smooth L1 Loss.
-   **Optimizer**: Adam with a learning rate of 0.01.
-   **Training**: Trained for 1000 epochs on the full 2016-2024 dataset and evaluated on the 2025 test set.

### 2. Neural Network (PyTorch)
-   **Architecture**: Multi-layered feed-forward neural network with ReLU activation and Dropout layers for regularization:
    -   `Linear(input_dim, 64)` -> `ReLU` -> `Dropout(0.2)`
    -   `Linear(64, 32)` -> `ReLU` -> `Dropout(0.2)`
    -   `Linear(32, 1)`
-   **Loss Functions**: MSE and RMSE.
-   **Optimizer**: Adam with a learning rate of 0.001.
-   **Training**: Trained for 1000 epochs on the full 2016-2024 dataset and evaluated on the 2025 test set.

### 3. Support Vector Machine (SVM)
-   **Model**: Support Vector Regressor (`SVR`) from scikit-learn.
-   **Initial Model**: `SVR(kernel='rbf', C=100, epsilon=0.1)`.
-   **Hyperparameter Tuning**: Performed using `GridSearchCV` with the custom walk-forward cross-validation strategy. The `param_grid` included:
    -   `C`: [0.1, 1, 10, 100]
    -   `kernel`: ['rbf', 'linear', 'poly', 'sigmoid']
    -   `gamma`: ['scale', 'auto']
    -   `epsilon`: [0.01, 0.1, 0.5]

## Results

| Model                  | Test RMSE  | Test MSE   | Best Hyperparameters (for Tuned SVM) |
| :--------------------- | :--------- | :--------- | :----------------------------------- |
| Linear Model           | 45.2947    | 2051.6138  | N/A                                  |
| Neural Network         | **7.4711** | 55.8173    | N/A                                  |
| Initial SVM Model      | 8.8844     | 78.9329    | N/A                                  |
| Tuned SVM Model        | 8.5149     | 72.5043    | {'C': 10, 'epsilon': 0.5, 'gamma': 'auto', 'kernel': 'rbf'} |


**Observation:**
The Neural Network model demonstrated the best performance, achieving the lowest Test RMSE of 7.4711. The tuned SVM model showed significant improvement over its initial configuration, reducing its Test RMSE from 8.8844 to 8.5149, confirming the effectiveness of hyperparameter tuning and walk-forward cross-validation. Both the Neural Network and Tuned SVM models significantly outperformed the simple Linear Model, highlighting the non-linear complexity of temperature forecasting.

## Future Work

-   **Explore more advanced models**: Investigate other time-series specific models like LSTMs, GRUs, or Prophet.
-   **External features**: Incorporate additional external features such as atmospheric pressure, humidity, and wind chill factors.
-   **Ensemble methods**: Experiment with ensemble techniques (e.g., stacking, bagging, boosting) to potentially improve prediction accuracy.
-   **Anomaly detection**: Implement anomaly detection techniques to identify and handle unusual weather events that might skew predictions.
-   **Real-time forecasting**: Develop a pipeline for real-time data ingestion and prediction to provide up-to-the-minute forecasts.


Dataset: Weather Station Records (42 stations, 2016-2025)
Columns: STATION, NAME, DATE, DAPR, MDPR, PRCP, SNOW, SNWD, TAVG, TMAX, TMIN, WDFG, WESD, WESF, WSFG

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

LOAD DATA

In [ ]:
df = pd.read_excel("4216555.xlsx", parse_dates=["DATE"])
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Loaded: 72,125 rows × 15 columns


,STATION,NAME,DATE,DAPR,MDPR,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG
0,CA00702FHL8,"STE ANNE DE BELLEVUE 1, QC CA",2016-01-01,NaN,NaN,0.04,NaN,3.5,29.0,32.0,26.0,27.0,NaN,NaN,87.2
1,CA00702S006,"MONTREAL PIERRE ELLIOTT TRUDEA, QC CA",2016-01-01,NaN,NaN,0.01,NaN,5.5,29.0,33.0,25.0,27.0,NaN,NaN,111.8
2,CA007017386,"ST JANVIER, QC CA",2016-01-01,NaN,NaN,0.00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CA007025251,"MONTREAL INTERNATIONAL A, QC CA",2016-01-01,NaN,NaN,0.02,0.5,7.1,28.0,32.0,25.0,26.0,NaN,NaN,116.3
4,CA00702LED4,"L ACADIE, QC CA",2016-01-01,NaN,NaN,0.02,NaN,7.5,29.0,32.0,26.0,0.0,NaN,NaN,0.0


### Daily Aggregation

This code aggregates weather data daily by identifying numeric columns, grouping by `DATE`, and calculating their means, storing the result in `daily`.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
daily = df.groupby("DATE")[numeric_cols].mean().reset_index()

print(f"\nAfter daily aggregation: ")
print(f"Loaded: {daily.shape[0]:,} rows × {daily.shape[1]} columns")
print(f"Date range: {daily['DATE'].min().date()} → {daily['DATE'].max().date()}")
print(f"\nMissing values after aggregation:")
print((daily.isnull().mean() * 100).round(1).to_string())
display(daily.head(6))


After daily aggregation: 
Loaded: 3,653 rows × 13 columns
Date range: 2016-01-01 → 2025-12-31

Missing values after aggregation:
DATE     0.0
DAPR    92.7
MDPR    92.7
PRCP     0.0
SNOW     0.1
SNWD     8.8
TAVG     6.9
TMAX     6.9
TMIN     6.9
WDFG    25.5
WESD    69.8
WESF    36.2
WSFG    26.2


,DATE,DAPR,MDPR,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG
0,2016-01-01,NaN,NaN,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,NaN,NaN,75.788889
1,2016-01-02,NaN,NaN,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,NaN,NaN,84.511111
2,2016-01-03,NaN,NaN,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,NaN,NaN,78.800000
3,2016-01-04,NaN,NaN,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,NaN,NaN,61.644444
4,2016-01-05,NaN,NaN,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,NaN,NaN,56.177778
5,2016-01-06,NaN,NaN,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,NaN,NaN,26.344444


##DROP EXTREMELY SPARSE COLUMNS (>90% missing)
This code removes columns with 90% or more missing data from the `daily` DataFrame. It calculates missing percentages, drops such columns. This cleans the dataset by removing sparse features.

In [ ]:
missing_pct = daily.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct >= 90].index.tolist()
daily.drop(columns=cols_to_drop, inplace=True)
print(f"\nDropped sparse columns (>90% missing): {cols_to_drop}")
print(f"Loaded: {daily.shape[0]:,} rows × {daily.shape[1]} columns")
display(daily.head(6))


Dropped sparse columns (>90% missing): ['DAPR', 'MDPR']
Loaded: 3,653 rows × 11 columns


,DATE,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG
0,2016-01-01,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,NaN,NaN,75.788889
1,2016-01-02,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,NaN,NaN,84.511111
2,2016-01-03,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,NaN,NaN,78.800000
3,2016-01-04,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,NaN,NaN,61.644444
4,2016-01-05,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,NaN,NaN,56.177778
5,2016-01-06,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,NaN,NaN,26.344444


##DATE FEATURE ENGINEERING
This code performs date feature engineering to capture seasonality. It extracts `year`, `month`, `day_of_year`, and `week` from the `DATE` column. Then, it applies cyclical transformations (sine and cosine) to `month` and `day_of_year` to create `month_sin`, `month_cos`, `doy_sin`, and `doy_cos` features. This encoding represents cyclic patterns without introducing arbitrary order. Finally, the original `DATE` column is dropped, and the new DataFrame's dimensions and initial rows are displayed.

In [ ]:
daily["year"]        = daily["DATE"].dt.year
daily["month"]       = daily["DATE"].dt.month
daily["day_of_year"] = daily["DATE"].dt.dayofyear
daily["week"]        = daily["DATE"].dt.isocalendar().week.astype(int)

daily["month_sin"] = np.sin(2 * np.pi * daily["month"] / 12)
daily["month_cos"] = np.cos(2 * np.pi * daily["month"] / 12)
daily["doy_sin"]   = np.sin(2 * np.pi * daily["day_of_year"] / 365)
daily["doy_cos"]   = np.cos(2 * np.pi * daily["day_of_year"] / 365)

daily.drop(columns=["DATE"], inplace=True)
print(f"Loaded: {daily.shape[0]:,} rows × {daily.shape[1]} columns")
display(daily.head(6))

Loaded: 3,653 rows × 18 columns


,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG,year,month,day_of_year,week,month_sin,month_cos,doy_sin,doy_cos
0,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,NaN,NaN,75.788889,2016,1,1,53,0.5,0.866025,0.017213,0.999852
1,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,NaN,NaN,84.511111,2016,1,2,53,0.5,0.866025,0.034422,0.999407
2,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,NaN,NaN,78.800000,2016,1,3,53,0.5,0.866025,0.051620,0.998667
3,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,NaN,NaN,61.644444,2016,1,4,1,0.5,0.866025,0.068802,0.997630
4,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,NaN,NaN,56.177778,2016,1,5,1,0.5,0.866025,0.085965,0.996298
5,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,NaN,NaN,26.344444,2016,1,6,1,0.5,0.866025,0.103102,0.994671


##DERIVED FEATURES
This code creates new features to enrich the dataset. It calculates `temp_range` as `TMAX` minus `TMIN` if both exist. It then creates binary flags: `snow_flag` is 1 if any `SNOW`, `prcp_flag` is 1 if any `PRCP`. These are added to the `daily` DataFrame, helping the model capture nuanced patterns.






In [ ]:
if "TMIN" in daily.columns and "TMAX" in daily.columns:
    daily["temp_range"] = daily["TMAX"] - daily["TMIN"]

if "SNOW" in daily.columns:
    daily["snow_flag"] = (daily["SNOW"] > 0).astype(int)

if "PRCP" in daily.columns:
    daily["prcp_flag"] = (daily["PRCP"] > 0).astype(int)

print(f"\nDerived features added. Total columns: {daily.shape[1]}")
print(f"Loaded: {daily.shape[0]:,} rows × {daily.shape[1]} columns")
display(daily.head(6))


Derived features added. Total columns: 21
Loaded: 3,653 rows × 21 columns


,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG,...,month,day_of_year,week,month_sin,month_cos,doy_sin,doy_cos,temp_range,snow_flag,prcp_flag
0,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,NaN,NaN,75.788889,...,1,1,53,0.5,0.866025,0.017213,0.999852,7.625000,1,1
1,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,NaN,NaN,84.511111,...,1,2,53,0.5,0.866025,0.034422,0.999407,9.470833,1,1
2,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,NaN,NaN,78.800000,...,1,3,53,0.5,0.866025,0.051620,0.998667,22.794872,1,1
3,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,NaN,NaN,61.644444,...,1,4,1,0.5,0.866025,0.068802,0.997630,16.233333,0,1
4,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,NaN,NaN,56.177778,...,1,5,1,0.5,0.866025,0.085965,0.996298,30.466667,0,0
5,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,NaN,NaN,26.344444,...,1,6,1,0.5,0.866025,0.103102,0.994671,24.838235,0,0


## IMPUTE MISSING VALUES
* Strategy: three-pass approach tailored for daily time-series weather data.
* No rows are dropped — every date is kept.

In [ ]:
numeric_daily_cols = daily.select_dtypes(include=[np.number]).columns.tolist()

### Pass 1 — Linear Interpolation (time-aware):
   Fills short gaps by drawing a straight line between the nearest known
  values before and after the gap. Best for temperature columns (TMAX, TMIN,
   TAVG) where values change gradually day-to-day.
   Example: if TMAX is [30, NaN, NaN, 36], interpolation fills [30, 32, 34, 36].

In [ ]:
daily[numeric_daily_cols] = daily[numeric_daily_cols].interpolate(
    method="linear", limit_direction="both"
)

### Pass 2 — Seasonal median (same calendar month, rolling 5-year window):
  For gaps that remain after interpolation (e.g. at the start/end of the series where there is no neighbor to interpolate from), we fill with the median value of the same month across the surrounding years. This respects seasonality: a missing January value is filled with a  January median, not an overall annual median.

In [ ]:
daily["_month"] = daily["day_of_year"].apply(
    lambda d: pd.Timestamp("2000-01-01") + pd.Timedelta(days=int(d) - 1)
).dt.month  # derive month from day_of_year since DATE was dropped

for col in numeric_daily_cols:
    mask = daily[col].isna()
    if mask.any():
        seasonal_median = daily.groupby("_month")[col].transform("median")
        daily.loc[mask, col] = seasonal_median[mask]

daily.drop(columns=["_month"], inplace=True)

### Pass 3 — Global median fallback:
  Any column that still has NaN after both passes (e.g. WDFG / WSFG which are sparse across the whole series) is filled with its overall median.  This is a last-resort catch-all to guarantee zero missing values.

In [ ]:
for col in numeric_daily_cols:
    if daily[col].isna().any():
        daily[col].fillna(daily[col].median(), inplace=True)

remaining = daily[numeric_daily_cols].isna().sum().sum()

In [ ]:
print(f"\nMissing values check:")
print((daily.isnull().mean() * 100).round(1).to_string())
print(f"Loaded: {daily.shape[0]:,} rows × {daily.shape[1]} columns")
display(daily.head(6))


Missing values check:
PRCP           0.0
SNOW           0.0
SNWD           0.0
TAVG           0.0
TMAX           0.0
TMIN           0.0
WDFG           0.0
WESD           0.0
WESF           0.0
WSFG           0.0
year           0.0
month          0.0
day_of_year    0.0
week           0.0
month_sin      0.0
month_cos      0.0
doy_sin        0.0
doy_cos        0.0
temp_range     0.0
snow_flag      0.0
prcp_flag      0.0
Loaded: 3,653 rows × 21 columns


,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG,...,month,day_of_year,week,month_sin,month_cos,doy_sin,doy_cos,temp_range,snow_flag,prcp_flag
0,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,0.3,0.3,75.788889,...,1,1,53,0.5,0.866025,0.017213,0.999852,7.625000,1,1
1,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,0.3,0.3,84.511111,...,1,2,53,0.5,0.866025,0.034422,0.999407,9.470833,1,1
2,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,0.3,0.3,78.800000,...,1,3,53,0.5,0.866025,0.051620,0.998667,22.794872,1,1
3,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,0.3,0.3,61.644444,...,1,4,1,0.5,0.866025,0.068802,0.997630,16.233333,0,1
4,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,0.3,0.3,56.177778,...,1,5,1,0.5,0.866025,0.085965,0.996298,30.466667,0,0
5,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,0.3,0.3,26.344444,...,1,6,1,0.5,0.866025,0.103102,0.994671,24.838235,0,0


##  DEFINE TARGET & FEATURES

 We shift TMAX forward by 1 day so that each row reads:
   X = today's weather observations
   y = tomorrow's maximum temperature

This reflects real-world prediction: at the end of today you have all of
 today's measurements and want to forecast what tomorrow's high will be.

 shift(-1) moves the TMAX column up by one row:
   Before shift:  row 0 → TMAX of Jan 1   (today)
   After shift:   row 0 → TMAX of Jan 2   (tomorrow)

 The very last row loses its target (no "tomorrow" exists) and is dropped.

In [ ]:
daily["TMAX_tomorrow"] = daily["TMAX"].shift(-1)

daily.dropna(subset=["TMAX_tomorrow"], inplace=True)

y = daily["TMAX_tomorrow"].values
X = daily.drop(columns=["TMAX_tomorrow"])

print(f"Loaded: {X.shape[0]:,} rows × {X.shape[1]} columns")
display(X.head(6))

print(f"Loaded: {y.shape[0]:,} rows × 1 columns")
display(y[:6])

Loaded: 3,652 rows × 21 columns


,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG,...,month,day_of_year,week,month_sin,month_cos,doy_sin,doy_cos,temp_range,snow_flag,prcp_flag
0,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,0.3,0.3,75.788889,...,1,1,53,0.5,0.866025,0.017213,0.999852,7.625000,1,1
1,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,0.3,0.3,84.511111,...,1,2,53,0.5,0.866025,0.034422,0.999407,9.470833,1,1
2,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,0.3,0.3,78.800000,...,1,3,53,0.5,0.866025,0.051620,0.998667,22.794872,1,1
3,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,0.3,0.3,61.644444,...,1,4,1,0.5,0.866025,0.068802,0.997630,16.233333,0,1
4,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,0.3,0.3,56.177778,...,1,5,1,0.5,0.866025,0.085965,0.996298,30.466667,0,0
5,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,0.3,0.3,26.344444,...,1,6,1,0.5,0.866025,0.103102,0.994671,24.838235,0,0


Loaded: 3,652 rows × 1 columns


array([32.53333333, 32.33333333,  7.73333333, 17.8       , 30.25      ,
       32.47368421])

## OUTLIER CLIPPING (1st–99th percentile)
 TMIN and temp_range had 74 values clipped symmetrically (37 on each side), which is expected for normally distributed weather data. Columns like PRCP, SNOW, WSFG only had values clipped above (0 below) — this makes physical sense since they can't go below zero, so the 1st percentile naturally lands at 0 and nothing gets clipped on the lower end.

In [ ]:
clipping_report = []
for col in X.select_dtypes(include=[np.number]).columns:
    lower = X[col].quantile(0.01)
    upper = X[col].quantile(0.99)
    n_lower = (X[col] < lower).sum()
    n_upper = (X[col] > upper).sum()
    if n_lower > 0 or n_upper > 0:
        clipping_report.append({
            "column"        : col,
            "lower_bound"   : round(lower, 2),
            "upper_bound"   : round(upper, 2),
            "clipped_below" : int(n_lower),
            "clipped_above" : int(n_upper),
            "total_clipped" : int(n_lower + n_upper),
        })
    X[col] = X[col].clip(lower=lower, upper=upper)

if clipping_report:
    report_df = pd.DataFrame(clipping_report).sort_values("total_clipped", ascending=False)
    print("\nOutlier clipping report:")
    print(report_df.to_string(index=False))
else:
    print("\nNo outliers clipped.")
print(f"Loaded: {daily.shape[0]:,} rows × {daily.shape[1]} columns")
display(daily.head(6))


Outlier clipping report:
     column  lower_bound  upper_bound  clipped_below  clipped_above  total_clipped
       TMAX         8.70        90.17             37             37             74
       TMIN       -10.31        69.67             37             37             74
 temp_range         4.19        34.22             37             37             74
       TAVG        -0.19        79.50             37             36             73
day_of_year         4.00       362.00             30             32             62
    doy_sin        -1.00         1.00             30             30             60
    doy_cos        -1.00         1.00             20             32             52
       WSFG         0.00       145.12              0             37             37
       PRCP         0.00         1.02              0             37             37
       SNOW         0.00         4.32              0             37             37
       SNWD         0.00        21.67              0         

,PRCP,SNOW,SNWD,TAVG,TMAX,TMIN,WDFG,WESD,WESF,WSFG,...,day_of_year,week,month_sin,month_cos,doy_sin,doy_cos,temp_range,snow_flag,prcp_flag,TMAX_tomorrow
0,0.021765,0.2200,8.786667,28.142857,32.125000,24.500000,19.888889,0.3,0.3,75.788889,...,1,53,0.5,0.866025,0.017213,0.999852,7.625000,1,1,32.533333
1,0.067333,1.0625,8.342857,27.785714,32.533333,23.062500,21.555556,0.3,0.3,84.511111,...,2,53,0.5,0.866025,0.034422,0.999407,9.470833,1,1,32.333333
2,0.150588,1.2000,8.292308,20.416667,32.333333,9.538462,25.111111,0.3,0.3,78.800000,...,3,53,0.5,0.866025,0.051620,0.998667,22.794872,1,1,7.733333
3,0.001000,0.0000,9.806667,-0.733333,7.733333,-8.500000,19.333333,0.3,0.3,61.644444,...,4,1,0.5,0.866025,0.068802,0.997630,16.233333,0,1,17.800000
4,0.000000,0.0000,9.543750,2.533333,17.800000,-12.666667,15.111111,0.3,0.3,56.177778,...,5,1,0.5,0.866025,0.085965,0.996298,30.466667,0,0,30.250000
5,0.000000,0.0000,9.053333,18.375000,30.250000,5.411765,7.333333,0.3,0.3,26.344444,...,6,1,0.5,0.866025,0.103102,0.994671,24.838235,0,0,32.473684


# WALK-FORWARD CROSS-VALIDATION SPLIT
Instead of one fixed train/val split, we use an expanding-window approach:
each fold trains on all data up to year N and validates on year N+1. This gives 4 independent val scores instead of 1, producing a much more robust estimate of how the model generalises to unseen future data.
Test set (2025) is held out completely and never used during fold evaluation.

*  Fold 1: Train [2016-2020] → Val [2021]
*  Fold 2: Train [2016-2021] → Val [2022]
*  Fold 3: Train [2016-2022] → Val [2023]
*  Fold 4: Train [2016-2023] → Val [2024]
*  Final : Train [2016-2024] → Test [2025]  (touched only once at the end)

In [ ]:
year_col = X["year"].values

X_test, y_test = X[year_col == 2025], y[year_col == 2025]

VAL_YEARS = [2021, 2022, 2023, 2024]
folds = []
for val_year in VAL_YEARS:
    train_mask = year_col < val_year
    val_mask   = year_col == val_year
    folds.append({
        "fold"       : len(folds) + 1,
        "train_years": f"2016-{val_year - 1}",
        "val_year"   : val_year,
        "X_train"    : X[train_mask],
        "y_train"    : y[train_mask],
        "X_val"      : X[val_mask],
        "y_val"      : y[val_mask],
    })

print("Walk-forward cross-validation folds:")
print(f"  {'Fold':<6} {'Train period':<15} {'Train rows':<12} {'Val year':<10} {'Val rows'}")
for f in folds:
    print(f"  {f['fold']:<6} {f['train_years']:<15} {len(f['X_train']):<12,} {f['val_year']:<10} {len(f['X_val']):,}")
print(f"  {'Final':<6} {'2016-2024':<15} {(year_col < 2025).sum():<12,} {'2025':<10} {len(X_test):,}  (test — held out)")

X_train_full = X[year_col < 2025]
y_train_full = y[year_col < 2025]

Walk-forward cross-validation folds:
  Fold   Train period    Train rows   Val year   Val rows
  1      2016-2020       1,827        2021       365
  2      2016-2021       2,192        2022       365
  3      2016-2022       2,557        2023       365
  4      2016-2023       2,922        2024       366
  Final  2016-2024       3,288        2025       364  (test — held out)


#  SCALE FEATURES
scaled_value = (original_value - mean) / std

Each fold fits its own scaler on that fold's training data only,
 preventing any leakage of future statistics into earlier folds.
 The final scaler is fit on all 2016-2024 data for the production model.

In [ ]:
for fold in folds:
    s = StandardScaler()
    fold["X_train_scaled"] = pd.DataFrame(s.fit_transform(fold["X_train"]), columns=X.columns)
    fold["X_val_scaled"]   = pd.DataFrame(s.transform(fold["X_val"]),       columns=X.columns)
    fold["scaler"]         = s

final_scaler       = StandardScaler()
X_train_scaled     = pd.DataFrame(final_scaler.fit_transform(X_train_full), columns=X.columns)
X_test_scaled      = pd.DataFrame(final_scaler.transform(X_test),           columns=X.columns)

print("Scalers fit per fold (train data only) + final scaler fit on 2016-2024")

Scalers fit per fold (train data only) + final scaler fit on 2016-2024


# Linear Model
Define a PyTorch linear model and implement Mean Squared Error (MSE) and Root Mean Squared Error (RMSE) loss functions using PyTorch tensor operations.


Define a linear model using PyTorch's `nn.Module` and implement Mean Squared Error (MSE) and Root Mean Squared Error (RMSE) using PyTorch tensor operations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LinearRegression(nn.Module):
    def __init__(self, input_dim):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.linear(x)

def mse_loss(y_pred, y_true):

    return F.mse_loss(y_pred, y_true)


def rmse_loss(y_pred, y_true):

    return torch.sqrt(mse_loss(y_pred, y_true))

def huber_loss(y_pred, y_true):

    return F.huber_loss(y_pred, y_true)

def smooth_l1_loss(y_pred, y_true):

    return F.smooth_l1_loss(y_pred, y_true)

## Train PyTorch Linear Model with final training - test dataset

Initialize a new PyTorch model, optimizer, and loss criterion for each fold. Since this is first try and a simple linear model, I decide to use final training data (2016-2024) and evaluate it on the held-out test set (2025), storing the training and test error histories.


In [ ]:
import torch.optim as optim

train_rmse_history = []
train_mse_history  = []
test_rmse_history  = []
test_mse_history   = []
train_huber_history = []
train_smooth_l1_history = []
test_huber_history = []
test_smooth_l1_history = []

input_dim = X_train_full.shape[1]

final_model = LinearRegression(input_dim)
final_optimizer = optim.Adam(final_model.parameters(), lr=0.01)

X_train_full_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
y_train_full_tensor = torch.tensor(y_train_full.reshape(-1, 1), dtype=torch.float32)
X_test_tensor       = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_test_tensor       = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

for epoch in range(1000):
    final_model.train()

    y_pred_train = final_model(X_train_full_tensor)

    current_train_mse  = mse_loss(y_pred_train, y_train_full_tensor).item()
    current_train_rmse = rmse_loss(y_pred_train, y_train_full_tensor).item()
    train_mse_history.append(current_train_mse)
    train_rmse_history.append(current_train_rmse)

    current_train_huber = huber_loss(y_pred_train, y_train_full_tensor).item()
    current_train_smooth_l1 = smooth_l1_loss(y_pred_train, y_train_full_tensor).item()
    train_huber_history.append(current_train_huber)
    train_smooth_l1_history.append(current_train_smooth_l1)

    final_optimizer.zero_grad()
    mse_loss(y_pred_train, y_train_full_tensor).backward()
    final_optimizer.step()

    final_model.eval()
    with torch.no_grad():
        y_pred_test = final_model(X_test_tensor)

        current_test_mse  = mse_loss(y_pred_test, y_test_tensor).item()
        current_test_rmse = rmse_loss(y_pred_test, y_test_tensor).item()
        test_mse_history.append(current_test_mse)
        test_rmse_history.append(current_test_rmse)

        current_test_huber = huber_loss(y_pred_test, y_test_tensor).item()
        current_test_smooth_l1 = smooth_l1_loss(y_pred_test, y_test_tensor).item()
        test_huber_history.append(current_test_huber)
        test_smooth_l1_history.append(current_test_smooth_l1)

print(f"\nFinal Training MSE: {train_mse_history[-1]:.4f}, Final Training RMSE: {train_rmse_history[-1]:.4f}")
print(f"Final Training Huber Loss: {train_huber_history[-1]:.4f}, Final Training Smooth L1 Loss: {train_smooth_l1_history[-1]:.4f}")
print(f"Final Test MSE: {test_mse_history[-1]:.4f}, Final Test RMSE: {test_rmse_history[-1]:.4f}")
print(f"Final Test Huber Loss: {test_huber_history[-1]:.4f}, Final Test Smooth L1 Loss: {test_smooth_l1_history[-1]:.4f}")


Final Training MSE: 1937.8192, Final Training RMSE: 44.0207
Final Training Huber Loss: 42.9659, Final Training Smooth L1 Loss: 42.9659
Final Test MSE: 2051.6138, Final Test RMSE: 45.2947
Final Test Huber Loss: 44.3228, Final Test Smooth L1 Loss: 44.3228


# Neural Network model
Train the PyTorch Neural Network model on the full training dataset (2016-2024) and evaluate its performance on the held-out test set (2025). The training should run for 1000 epochs, and the training and test MSE/RMSE errors should be stored at each epoch. Use the `X_train_scaled`, `y_train_full`, `X_test_scaled`, and `y_test` datasets, and the `mse_loss` and `rmse_loss` functions already defined.

## Define PyTorch Neural Network with Regularization

Define a multi-layered neural network using PyTorch's `nn.Module`, including multiple linear layers, ReLU activation functions, and Dropout layers for regularization. The `mse_loss` and `rmse_loss` functions will be reused.


In [ ]:
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNetwork, self).__init__()

        self.fc1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.2)


        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)

        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

print("PyTorch NeuralNetwork model with regularization defined.")

PyTorch NeuralNetwork model with regularization defined.


In [ ]:
import torch.optim as optim

train_nn_rmse_history = []
train_nn_mse_history  = []
test_nn_rmse_history  = []
test_nn_mse_history   = []

input_dim = X_train_full.shape[1]

neural_net_model = NeuralNetwork(input_dim)

optimizer_nn = optim.Adam(neural_net_model.parameters(), lr=0.001)

X_train_full_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
y_train_full_tensor = torch.tensor(y_train_full.reshape(-1, 1), dtype=torch.float32)
X_test_tensor       = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_test_tensor       = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

for epoch in range(1000):
    neural_net_model.train()

    y_pred_train_nn = neural_net_model(X_train_full_tensor)

    current_train_nn_mse  = mse_loss(y_pred_train_nn, y_train_full_tensor).item()
    current_train_nn_rmse = rmse_loss(y_pred_train_nn, y_train_full_tensor).item()
    train_nn_mse_history.append(current_train_nn_mse)
    train_nn_rmse_history.append(current_train_nn_rmse)

    optimizer_nn.zero_grad()
    mse_loss(y_pred_train_nn, y_train_full_tensor).backward()
    optimizer_nn.step()

    neural_net_model.eval()
    with torch.no_grad():
        y_pred_test_nn = neural_net_model(X_test_tensor)

        current_test_nn_mse  = mse_loss(y_pred_test_nn, y_test_tensor).item()
        current_test_nn_rmse = rmse_loss(y_pred_test_nn, y_test_tensor).item()
        test_nn_mse_history.append(current_test_nn_mse)
        test_nn_rmse_history.append(current_test_nn_rmse)

print(f"\nFinal Neural Network Training MSE: {train_nn_mse_history[-1]:.4f}, Final Training RMSE: {train_nn_rmse_history[-1]:.4f}")
print(f"Final Neural Network Test MSE: {test_nn_mse_history[-1]:.4f}, Final Test RMSE: {test_nn_rmse_history[-1]:.4f}")


Final Neural Network Training MSE: 109.1060, Final Training RMSE: 10.4454
Final Neural Network Test MSE: 55.8173, Final Test RMSE: 7.4711


## Observation of Linear Model and Neural Network

Linear Model: Achieved a Test RMSE of 45.2947.
Neural Network: Achieved a significantly lower Test RMSE of 7.4711.
Observation: The Neural Network model demonstrated substantially better performance than the simpler Linear Model. The Test RMSE for the Neural Network is much lower (7.4711) compared to the Linear Model (45.2947). This indicates that the non-linear capabilities and multi-layered architecture of the Neural Network were far more effective in capturing the complex patterns in the weather data to predict maximum temperatures.

## Support Vector Machine (SVM) Model

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
import numpy as np

svm_model = SVR(kernel='rbf', C=100, epsilon=0.1)
svm_model.fit(X_train_scaled, y_train_full)

y_pred_train_svm = svm_model.predict(X_train_scaled)

y_pred_test_svm = svm_model.predict(X_test_scaled)

mse_train_svm = mean_squared_error(y_train_full, y_pred_train_svm)
rmse_train_svm = np.sqrt(mse_train_svm)

mse_test_svm = mean_squared_error(y_test, y_pred_test_svm)
rmse_test_svm = np.sqrt(mse_test_svm)

print(f"\nSVM Training MSE: {mse_train_svm:.4f}, SVM Training RMSE: {rmse_train_svm:.4f}")
print(f"SVM Test MSE: {mse_test_svm:.4f}, SVM Test RMSE: {rmse_test_svm:.4f}")


Training SVM model...
SVM model trained.

SVM Training MSE: 25.0140, SVM Training RMSE: 5.0014
SVM Test MSE: 78.9329, SVM Test RMSE: 8.8844


##Comparison with previous models:

Compared to the Linear Model: The SVM model shows a substantial improvement. The Linear Model had a test RMSE of 45.2947, while the SVM achieved a significantly lower test RMSE of 8.8844. This indicates that the SVM, with its non-linear capabilities, is much better at capturing the relationships in the data than the simple linear model.

Compared to the Neural Network, the Neural Network achieved a test RMSE of 7.4711, slightly better than the SVM's 8.8844. However, the SVM's training RMSE (5.0014) is lower than the Neural Network's training RMSE (10.4454), but its test RMSE is higher. This could suggest that while the SVM is fitting the training data very closely, the Neural Network's architecture and regularization (like Dropout) might be leading to better generalization on unseen data, despite a higher training error.

##Perform hyperparameter tuning for the SVM model
We will use `GridSearchCV` with the walk-forward cross-validation strategy.

First, we will define a parameter grid for the `SVR` model, including common hyperparameters like `C`, `kernel`, `gamma`, and `epsilon`. Next, we will create a custom cross-validation iterator that mirrors the walk-forward folds previously established (training on expanding windows of years and validating on the subsequent year). This ensures a robust evaluation of hyperparameters consistent with our time series approach. We will then initialize `GridSearchCV` with the `SVR` estimator, the parameter grid, the custom cross-validation strategy, and `neg_mean_squared_error` as the scoring metric.

After configuring, we will fit the `GridSearchCV` object to the full-scaled training data (`X_train_scaled`, `y_train_full`) to identify the optimal hyperparameters. Finally, we will retrieve the best `SVR` model, predict on the held-out test set (`X_test_scaled`), and calculate its Mean Squared Error (`MSE`) and Root Mean Squared Error (`RMSE`). We will then print the best hyperparameters found and the test MSE and RMSE to assess the model's performance.

## Define Parameter Grid for SVM

Define a dictionary of hyperparameters and their possible values for the SVM model (e.g., 'C', 'kernel', 'gamma', 'epsilon').


In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.01, 0.1, 0.5]
}

print("Parameter grid for SVM defined:")
print(param_grid)

Parameter grid for SVM defined:
{'C': [0.1, 1, 10, 100], 'kernel': ['rbf', 'linear', 'poly', 'sigmoid'], 'gamma': ['scale', 'auto'], 'epsilon': [0.01, 0.1, 0.5]}


## Configure GridSearchCV

Initialize GridSearchCV with the SVR model, the defined parameter grid, an appropriate scoring metric (e.g., 'neg_mean_squared_error'), and a custom walk-forward cross-validation strategy.


In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV

custom_cv_splits = []

years_in_X_train_full = X_train_full['year'].values

for fold_data in folds:
    val_year = fold_data['val_year']

    train_indices = np.where(years_in_X_train_full < val_year)[0]

    val_indices = np.where(years_in_X_train_full == val_year)[0]

    if len(train_indices) > 0 and len(val_indices) > 0:
        custom_cv_splits.append((train_indices, val_indices))

svr_estimator = SVR()

grid_search = GridSearchCV(
    estimator=svr_estimator,
    param_grid=param_grid,
    cv=custom_cv_splits,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=2
)

print("GridSearchCV initialized with SVR and custom walk-forward cross-validation.")

GridSearchCV initialized with SVR and custom walk-forward cross-validation.


In [ ]:
print("Starting GridSearchCV for SVR...")

grid_search.fit(X_train_scaled, y_train_full.ravel())

print("GridSearchCV for SVR completed.")

Starting GridSearchCV for SVR...
Fitting 4 folds for each of 96 candidates, totalling 384 fits
GridSearchCV for SVR completed.


In [ ]:
from sklearn.metrics import mean_squared_error

best_svr_model = grid_search.best_estimator_

y_pred_test_best_svm = best_svr_model.predict(X_test_scaled)

mse_test_best_svm = mean_squared_error(y_test, y_pred_test_best_svm)
rmse_test_best_svm = np.sqrt(mse_test_best_svm)

print("\n--- SVR Hyperparameter Tuning Results ---")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best cross-validation score (neg_mean_squared_error): {grid_search.best_score_:.4f}")
print(f"Test MSE of Best SVR Model: {mse_test_best_svm:.4f}")
print(f"Test RMSE of Best SVR Model: {rmse_test_best_svm:.4f}")

print("\n--- Comparison with previous models (from the notebook) ---")
print(f"Linear Model Test RMSE: {test_rmse_history[-1]:.4f}")
print(f"Neural Network Test RMSE: {test_nn_rmse_history[-1]:.4f}")
print(f"Initial SVM Model Test RMSE: {rmse_test_svm:.4f}")

if rmse_test_best_svm < rmse_test_svm:
    print(f"\nHyperparameter tuning *improved* the SVM model's test RMSE from {rmse_test_svm:.4f} to {rmse_test_best_svm:.4f}.")
elif rmse_test_best_svm > rmse_test_svm:
    print(f"\nHyperparameter tuning *did not improve* the SVM model's test RMSE, changing from {rmse_test_svm:.4f} to {rmse_test_best_svm:.4f}.")
else:
    print(f"\nHyperparameter tuning resulted in similar SVM model test RMSE as before: {rmse_test_best_svm:.4f}.")


--- SVR Hyperparameter Tuning Results ---
Best Hyperparameters: {'C': 10, 'epsilon': 0.5, 'gamma': 'auto', 'kernel': 'rbf'}
Best cross-validation score (neg_mean_squared_error): -46.2260
Test MSE of Best SVR Model: 72.5043
Test RMSE of Best SVR Model: 8.5149

--- Comparison with previous models (from the notebook) ---
Linear Model Test RMSE: 45.2947
Neural Network Test RMSE: 7.4711
Initial SVM Model Test RMSE: 8.8844

Hyperparameter tuning *improved* the SVM model's test RMSE from 8.8844 to 8.5149.


## Obeservation

Compared to other models:
The tuned SVR model (8.5149 RMSE) outperformed the initial SVM model (8.8844 RMSE) and the Linear Model (45.2947 RMSE).
However, the tuned SVR model did not surpass the Neural Network model, which achieved a lower Test RMSE of 7.4711.
The walk-forward cross-validation strategy effectively identified optimal hyperparameters for the SVR model, leading to a notable improvement over its untuned version, which highlights the value of proper time-series specific validation.
While tuning improved the SVR model, it still lags behind the Neural Network model in terms of RMSE performance.
